In [1]:
from langchain_core.messages import HumanMessage, SystemMessage
# 举例1
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint
from pydantic import BaseModel, Field
from llm.my_llm import model_tool

class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str=Field(description='用户到姓名')
    email:str=Field(description='用户到邮箱')
    phone:str=Field(description='用户到电话')

# 用 ChatOpenAI 走 DashScope 兼容接口，extra_body 才能正确传递


agent=create_agent(
    model=model_tool,
    response_format=ToolStrategy(ContractInfo)
)

res=agent.invoke({
    "messages":[{'role':'user','content':'从以下信息中提取用户消息，小明的邮箱是asdasdad@163.com,电话是19966155502'}]
})
# rprint(res)
print(res['structured_response'])

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint
from pydantic import BaseModel, Field
from llm.my_llm import model_tool

class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str=Field(description='用户到姓名')
    email:str=Field(description='用户到邮箱')
    phone:str=Field(description='用户到电话')

agent=create_agent(
    model=model_tool,
    response_format=ToolStrategy(ContractInfo)
)

res=agent.invoke({
    "messages":[
        HumanMessage('从以下信息中提取用户消息，小明的邮箱是asdasdad@163.com,电话是19966155502')
    ]
})
rprint(res)
# print(res['structured_response'])

name='小明' email='asdasdad@163.com' phone='19966155502'


{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户消息，小明的邮箱是asdasdad@163.com,电话是19966155502',
            additional_kwargs={},
            response_metadata={},
            id='ca922c30-4ef3-41cc-b3ba-63c4a698304e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 62,
                    'prompt_tokens': 350,
                    'total_tokens': 412,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': None,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 62
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 350}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.6-flash-2026-04-16',
                'system_fingerprint': None,
                'id': 'chatcmpl-e1ad36b4-8db6-9493-af7f-fa5c679ab465',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3cc7-4a2b-7ac3-99f9-36a7d200f1c1-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'asdasdad@163.com', 'phone': '19966155502'},
                    'id': 'call_5799c4a832124774ae986f6d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 350,
                'output_tokens': 62,
                'total_tokens': 412,
                'input_token_details': {},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='asdasdad@163.com' phone='19966155502'",
            name='ContractInfo',
            id='533621f4-946b-479e-950b-f374cfe20b7a',
            tool_call_id='call_5799c4a832124774ae986f6d'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='asdasdad@163.com', phone='19966155502')
}

In [5]:
from typing import Literal, TypedDict, Annotated
# 举例2:添加工具的调用
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint
from pydantic import BaseModel, Field
from llm.my_llm import model_tool
from langchain.tools import tool

# 自定义工具
@tool(parse_docstring=True)
def search_customer_database(query:str)->str:
    """
        在客户数据库中检索数据

        Args:
            query : 客户查询字符串，例如"张三"或"李四"

        Return:
            str : 客户记录字符串，包含客户姓名、等级、最近购买日期和累积消费
    """
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累积消费：¥15,000"

    elif "李四" in query.lower():
        return  "客户记录：李四，VIP客户，最近购买日期：2026-12-15，累积消费：¥32,000"
    else:
        return f"关于客户{query},无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

agent=create_agent(
    model=model_tool,
    system_prompt=SystemMessage(""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"),
    response_format=ToolStrategy(CustomerAnalysis),
    tools=[search_customer_database, send_email]
)
res=agent.invoke({
    "messages":[HumanMessage("请分析客户李四")]
})
rprint(res)



{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='d03af67b-b96f-4897-8545-df7355e84b55'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 609,
                    'total_tokens': 632,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': None,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 23
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 609}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.6-flash-2026-04-16',
                'system_fingerprint': None,
                'id': 'chatcmpl-162422d6-2e3f-9d22-a5eb-ac800693ba51',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3cda-6439-7011-9374-659aad0db3f7-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_53836a80620340b1a58d177a',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 609,
                'output_tokens': 23,
                'total_tokens': 632,
                'input_token_details': {},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：李四，VIP客户，最近购买日期：2026-12-15，累积消费：¥32,000',
            name='search_customer_database',
            id='4fd736fe-0b3c-4371-bfe6-9544b98a48b8',
            tool_call_id='call_53836a80620340b1a58d177a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 121,
                    'prompt_tokens': 687,
                    'total_tokens': 808,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': None,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 121
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 687}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.6-flash-2026-04-16',
                'system_fingerprint': None,
                'id': 'chatcmpl-b555dd5f-085f-94b3-b41e-614b59c84f08',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3cda-65fb-78b3-ac59-e9ec010c2c00-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '李四'},
                    'id': 'call_c669ca8a93e84b58954190d3',
                    'type': 'tool_call'
                },
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': 'VIP客户',
                        'recent_activity': '最近购买日期：2026-12-15',
                        'spending_level': '高',
                        'send_email': True
                    },
                    'id': 'call_74fd7b9b76714f9c9b2ec67c',
                    'type': 'tool_call'
     

### 使用TypedDict

In [3]:
"""
    2、使用ToolStrategy结构化输出
    注意：ToolStrategy 会设置 tool_choice="required"，
    通义千问 thinking 模式不支持该参数，必须用 ChatOpenAI 走兼容接口并关闭 thinking
"""
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint
from pydantic import BaseModel, Field
from llm.my_llm import model_tool
from typing import Literal, TypedDict, Annotated

class ContractInfo(TypedDict):
    """用户的联系方式"""
    name: Annotated[str, ..., '用户到姓名']    # ...表示必填
    email: Annotated[str, ..., '用户到邮箱']
    phone: Annotated[str, '用户到电话']

# 用 ChatOpenAI 走 DashScope 兼容接口，extra_body 才能正确传递


agent=create_agent(
    model=model_tool,
    response_format=ToolStrategy(ContractInfo)
)

res=agent.invoke({
    "messages":[{'role':'user','content':'从以下信息中提取用户消息，小明的邮箱是asdasdad@163.com,电话是19966155502'}]
})
# rprint(res)
print(res['structured_response'])

{'name': '小明', 'email': 'asdasdad@163.com', 'phone': '19966155502'}
